# 10. Advanced SQL Analysis

## Purpose

This notebook demonstrates advanced SQL techniques using the analytical
tables created in the previous stages of the E-commerce Business Analytics
project.

## Topics Covered

- Common Table Expressions (CTEs)
- JOINs
- Subqueries
- CASE statements
- Window functions
- ROW_NUMBER()
- RANK()
- DENSE_RANK()
- LAG()
- LEAD()
- Running totals
- Percentage calculations
- Customer segmentation
- Product ranking
- Time-based analysis

## Objective

Transform the Gold-layer data into meaningful analytical results
using advanced SQL techniques and answer important business questions.

#####First SQL: CTE
#####Which products generated the highest total sales value?

In [0]:
%sql

WITH product_sales AS (

    SELECT
        product_id,
        SUM(total_sales_value) AS total_sales_value
    FROM gold_product_analysis
    GROUP BY product_id

)

SELECT
    product_id,
    ROUND(total_sales_value, 2) AS total_sales_value
FROM product_sales
ORDER BY total_sales_value DESC
LIMIT 10;

#####CTE + JOIN
#####Which customers have the highest total spending and how many orders did they place?

In [0]:
%sql

WITH customer_summary AS (

    SELECT
        customer_id,
        total_orders,
        total_spending
    FROM gold_customer_analysis

)

SELECT
    c.customer_id,
    c.total_orders,
    ROUND(c.total_spending, 2) AS total_spending
FROM customer_summary c
ORDER BY c.total_spending DESC
LIMIT 10;

#####ROW_NUMBER() — Window Function
#####Rank customers based on their total spending.

In [0]:
%sql

SELECT
    customer_id,
    total_orders,
    ROUND(total_spending, 2) AS total_spending,

    ROW_NUMBER() OVER (
        ORDER BY total_spending DESC
    ) AS spending_rank

FROM gold_customer_analysis

ORDER BY spending_rank
LIMIT 10;

#####RANK() — Product Sales Ranking

#####Rank products based on their total sales value using the RANK() window function.

In [0]:
%sql

SELECT
    product_id,
    total_orders,
    total_order_items,
    ROUND(total_sales_value, 2) AS total_sales_value,

    RANK() OVER (
        ORDER BY total_sales_value DESC
    ) AS sales_rank

FROM gold_product_analysis

ORDER BY sales_rank

LIMIT 10;

##### DENSE_RANK() — Customer Spending

#####Rank customers based on total spending using DENSE_RANK().

In [0]:
%sql

SELECT
    customer_id,
    total_orders,
    ROUND(total_spending, 2) AS total_spending,

    DENSE_RANK() OVER (
        ORDER BY total_spending DESC
    ) AS spending_rank

FROM gold_customer_analysis

ORDER BY spending_rank

LIMIT 10;

##### Running Total — Monthly Sales

#####Calculate monthly sales and the cumulative sales value over time.

#####This demonstrates the use of SUM() as a window function.

In [0]:
%sql

WITH monthly_sales AS (
    SELECT
        YEAR(d.order_purchase_timestamp) AS year,
        MONTH(d.order_purchase_timestamp) AS month,
        ROUND(SUM(f.price + f.freight_value), 2) AS monthly_sales
    FROM fact_sales f
    INNER JOIN dim_order d
        ON f.order_id = d.order_id
    GROUP BY
        YEAR(d.order_purchase_timestamp),
        MONTH(d.order_purchase_timestamp)
)

SELECT
    year,
    month,
    monthly_sales,

    ROUND(
        SUM(monthly_sales) OVER (
            ORDER BY year, month
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ),
        2
    ) AS running_total_sales

FROM monthly_sales

ORDER BY year, month;

##### LAG() — Month-over-Month Sales

#####Compare each month's sales with the previous month's sales using the LAG() window function.

#####This helps identify how sales change from one month to the next.

In [0]:
%sql

WITH monthly_sales AS (
    SELECT
        YEAR(d.order_purchase_timestamp) AS year,
        MONTH(d.order_purchase_timestamp) AS month,
        ROUND(SUM(f.price + f.freight_value), 2) AS monthly_sales
    FROM fact_sales f
    INNER JOIN dim_order d
        ON f.order_id = d.order_id
    GROUP BY
        YEAR(d.order_purchase_timestamp),
        MONTH(d.order_purchase_timestamp)
),

sales_with_previous AS (
    SELECT
        year,
        month,
        monthly_sales,

        LAG(monthly_sales) OVER (
            ORDER BY year, month
        ) AS previous_month_sales

    FROM monthly_sales
)

SELECT
    year,
    month,
    monthly_sales,
    ROUND(previous_month_sales, 2) AS previous_month_sales,

    ROUND(
        monthly_sales - previous_month_sales,
        2
    ) AS sales_change

FROM sales_with_previous

ORDER BY year, month;

#####LEAD()

In [0]:
%sql

WITH monthly_sales AS (
    SELECT
        YEAR(d.order_purchase_timestamp) AS year,
        MONTH(d.order_purchase_timestamp) AS month,
        ROUND(SUM(f.price + f.freight_value), 2) AS monthly_sales
    FROM fact_sales f
    JOIN dim_order d
        ON f.order_id = d.order_id
    GROUP BY
        YEAR(d.order_purchase_timestamp),
        MONTH(d.order_purchase_timestamp)
)

SELECT
    year,
    month,
    monthly_sales,

    LEAD(monthly_sales) OVER (
        ORDER BY year, month
    ) AS next_month_sales,

    ROUND(
        LEAD(monthly_sales) OVER (
            ORDER BY year, month
        ) - monthly_sales,
        2
    ) AS sales_change_next_month

FROM monthly_sales

ORDER BY year, month;

#####CASE Statement

In [0]:
%sql

SELECT
    customer_id,
    total_orders,
    ROUND(total_spending, 2) AS total_spending,

    CASE
        WHEN total_spending >= 10000 THEN 'High Value'
        WHEN total_spending >= 5000 THEN 'Medium Value'
        ELSE 'Low Value'
    END AS customer_category

FROM gold_customer_analysis

ORDER BY total_spending DESC
LIMIT 20;

#####Subquery

In [0]:
%sql

SELECT
    customer_id,
    total_orders,
    ROUND(total_spending, 2) AS total_spending
FROM gold_customer_analysis
WHERE total_spending > (
    SELECT AVG(total_spending)
    FROM gold_customer_analysis
)
ORDER BY total_spending DESC
LIMIT 20;

#####Percentage Analysis

In [0]:
%sql

SELECT
    product_id,
    ROUND(total_sales_value, 2) AS total_sales_value,

    ROUND(
        total_sales_value * 100.0 /
        SUM(total_sales_value) OVER (),
        2
    ) AS sales_percentage

FROM gold_product_analysis

ORDER BY total_sales_value DESC
LIMIT 20;

In [0]:
%sql

WITH product_sales AS (
    SELECT
        product_id,
        ROUND(total_sales_value, 2) AS total_sales_value
    FROM gold_product_analysis
)

SELECT
    product_id,
    total_sales_value,

    ROUND(
        total_sales_value * 100.0 /
        SUM(total_sales_value) OVER (),
        2
    ) AS sales_percentage,

    ROUND(
        SUM(total_sales_value) OVER (
            ORDER BY total_sales_value DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) * 100.0 /
        SUM(total_sales_value) OVER (),
        2
    ) AS cumulative_sales_percentage

FROM product_sales

ORDER BY total_sales_value DESC
LIMIT 20;